# LeetCode 1658 — Minimum Operations to Reduce X to Zero bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-reduce-x-to-zero/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Sliding Window, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int], x: int) -> int:
        ...
```

Notebook đi từ **thử mọi cặp (lấy `l` phần tử bên trái, `r` phần tử bên phải)** → **bảng băm tổng hậu tố** → **tìm kiếm nhị phân** trên tổng hậu tố → **đảo bài toán**: tìm **đoạn con liên tiếp dài nhất** có tổng `sum(nums) − x` bằng **cửa sổ trượt** (lời giải chính) hoặc bằng **tổng tiền tố + bảng băm**.

## 1. Mục tiêu học

1. Mô hình hóa: sau mọi thao tác, phần bị lấy đi là một **tiền tố** + một **hậu tố**; phần còn lại là một **đoạn con liên tiếp**.
2. Biết "**đảo ngược bài toán**": tối thiểu số phần tử lấy đi ⇔ tối đa độ dài đoạn giữa có tổng `total − x`.
3. Thành thạo **cửa sổ trượt** cho mảng **số dương** với tổng mục tiêu, và phiên bản **tổng tiền tố + hash map** (dùng được cả khi có số âm).
4. Xử lý đúng các trường hợp biên: `x > total` (vô nghiệm), `x == total` (lấy hết), không tìm được đoạn (trả `−1`).
5. Đối chiếu nhiều cách với vét cạn trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `x`. Mỗi **thao tác**, bạn lấy ra phần tử **ngoài cùng bên trái** hoặc **ngoài cùng bên phải** của `nums` và **trừ** giá trị của nó khỏi `x`. Mảng bị thay đổi cho các thao tác sau.

Trả về **số thao tác ít nhất** để `x` trở thành **đúng 0**; nếu không thể, trả về `−1`.

**Ví dụ 1:** `nums = [1,1,4,2,3]`, `x = 5` → `2` (lấy hai phần tử cuối `3`, `2`).

**Ví dụ 2:** `nums = [5,6,7,8,9]`, `x = 4` → `−1`.

**Ví dụ 3:** `nums = [3,2,20,1,1,3]`, `x = 10` → `5` (lấy ba phần tử cuối và hai phần tử đầu: `3+2+1+1+3 = 10`).

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ 10⁴` (**số dương**)
- `1 ≤ x ≤ 10⁹`

In [1]:
import bisect
import random
import time
from itertools import accumulate
from typing import List

examples = [([1, 1, 4, 2, 3], 5, 2), ([5, 6, 7, 8, 9], 4, -1), ([3, 2, 20, 1, 1, 3], 10, 5)]

def apply_plan(nums, x, plan):
    """plan: chuỗi 'L'/'R'. Trả về x còn lại sau khi lấy lần lượt."""
    arr = list(nums)
    for p in plan:
        x -= arr.pop(0) if p == 'L' else arr.pop()
    return x

print("Ví dụ 1, lấy 'RR':", apply_plan(*examples[0][:2], "RR"))
print("Ví dụ 3, lấy 'RRRLL':", apply_plan(*examples[2][:2], "RRRLL"))
print("Ví dụ 3, thứ tự không quan trọng 'LRLRR':", apply_plan(*examples[2][:2], "LRLRR"))

Ví dụ 1, lấy 'RR': 0
Ví dụ 3, lấy 'RRRLL': 0
Ví dụ 3, thứ tự không quan trọng 'LRLRR': 0


## 3. Kiến thức nền

### 3.1 Tiền tố và hậu tố của mảng

Lấy `l` phần tử bên trái và `r` phần tử bên phải (với `l + r ≤ n`) ⇔ lấy `nums[:l]` và `nums[n−r:]`. **Thứ tự** các thao tác không ảnh hưởng tổng — chỉ `(l, r)` quan trọng.

In [2]:
nums = [3, 2, 20, 1, 1, 3]
l, r = 2, 3
n = len(nums)
print("trái :", nums[:l], "| phải:", nums[n - r:], "| còn lại:", nums[l:n - r])
print("tổng lấy đi =", sum(nums[:l]) + sum(nums[n - r:]))

trái : [3, 2] | phải: [1, 1, 3] | còn lại: [20]
tổng lấy đi = 10


### 3.2 Tổng tiền tố với `itertools.accumulate`

`pre[k] = nums[0] + … + nums[k−1]`, `pre[0] = 0`. Tổng đoạn `nums[i:j]` = `pre[j] − pre[i]`. Tổng hậu tố dài `r` = `total − pre[n − r]`.

In [3]:
pre = list(accumulate(nums, initial=0))
total = pre[-1]
print("pre =", pre, "| total =", total)
print("tổng 3 phần tử cuối =", total - pre[n - 3])

pre = [0, 3, 5, 25, 26, 27, 30] | total = 30
tổng 3 phần tử cuối = 5


### 3.3 `dict` tra ngược "tổng → chỉ số"

Lưu `pos[tổng] = chỉ số` để hỏi "có tiền tố/hậu tố nào đạt tổng `t` không, và dài bao nhiêu" trong `O(1)`.

In [4]:
suffix_len = {}
s = 0
suffix_len[0] = 0
for r in range(1, n + 1):
    s += nums[n - r]
    suffix_len[s] = r
print("tổng hậu tố -> độ dài:", suffix_len)

tổng hậu tố -> độ dài: {0: 0, 3: 1, 4: 2, 5: 3, 25: 4, 27: 5, 30: 6}


### 3.4 `bisect` trên mảng tăng dần

Vì `nums[i] > 0`, các tổng hậu tố **tăng nghiêm ngặt** theo độ dài ⇒ có thể tìm kiếm nhị phân bằng `bisect_left`.

In [5]:
suf = [0]
for v in reversed(nums):
    suf.append(suf[-1] + v)
print("tổng hậu tố theo độ dài:", suf)
k = bisect.bisect_left(suf, 7)
print("độ dài hậu tố có tổng 7:", k if k < len(suf) and suf[k] == 7 else None)

tổng hậu tố theo độ dài: [0, 3, 4, 5, 25, 27, 30]
độ dài hậu tố có tổng 7: None


## 4. Ý tưởng nền tảng / trực giác

Phần bị lấy đi = tiền tố + hậu tố ⇒ phần **còn lại** là một **đoạn liên tiếp ở giữa**. Gọi `total = sum(nums)`:

`tổng lấy đi = x` ⇔ `tổng đoạn giữa = total − x`

Muốn **lấy đi ít nhất** ⇔ muốn **đoạn giữa dài nhất**.

Tính tay ví dụ 3: `nums = [3,2,20,1,1,3]`, `total = 30`, `x = 10` ⇒ cần đoạn giữa có tổng `20`. Các đoạn tổng 20: `[20]` (dài 1). ⇒ lấy đi `6 − 1 = 5` phần tử.

Ví dụ 1: `total = 11`, `x = 5` ⇒ đoạn tổng `6`: `[1,1,4]` (dài 3), `[4,2]` (dài 2) ⇒ dài nhất 3 ⇒ lấy `5 − 3 = 2`.

Ví dụ 2: `total = 35`, `x = 4` ⇒ đoạn tổng `31`: không có ⇒ `−1`.

In [6]:
def all_middle(nums, target):
    n = len(nums)
    out = []
    for i in range(n + 1):
        for j in range(i, n + 1):
            if sum(nums[i:j]) == target:
                out.append((i, j, nums[i:j]))
    return out

for nums_, x_, exp in examples:
    target = sum(nums_) - x_
    segs = all_middle(nums_, target)
    best = max((j - i for i, j, _ in segs), default=None)
    print(f"nums={nums_}, x={x_}: cần đoạn tổng {target}; các đoạn: {[s for _, _, s in segs]}; dài nhất {best} -> đáp án {len(nums_) - best if best is not None else -1}")

nums=[1, 1, 4, 2, 3], x=5: cần đoạn tổng 6; các đoạn: [[1, 1, 4], [4, 2]]; dài nhất 3 -> đáp án 2
nums=[5, 6, 7, 8, 9], x=4: cần đoạn tổng 31; các đoạn: []; dài nhất None -> đáp án -1
nums=[3, 2, 20, 1, 1, 3], x=10: cần đoạn tổng 20; các đoạn: [[20]]; dài nhất 1 -> đáp án 5


## 5. Cách 1 — Brute Force / baseline: thử mọi `(l, r)`

**Ý tưởng:** thử mọi số phần tử lấy bên trái `l` và bên phải `r` (với `l + r ≤ n`), kiểm tra `pre[l] + suf[r] == x`, lấy `min(l + r)`.

In [7]:
def min_ops_brute(nums, x):
    n = len(nums)
    pre = list(accumulate(nums, initial=0))
    total = pre[-1]
    best = float('inf')
    for l in range(n + 1):
        for r in range(n - l + 1):
            if pre[l] + (total - pre[n - r]) == x:
                best = min(best, l + r)
    return best if best != float('inf') else -1

for nums_, x_, exp in examples:
    print(min_ops_brute(nums_, x_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
5 (kỳ vọng 5 )


**Độ phức tạp:** `O(n²)` cặp `(l, r)` — với `n = 10⁵` là `5·10⁹`: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** với mỗi `l`, ta cần **đúng một** giá trị tổng hậu tố `x − pre[l]` — có thể **tra cứu** thay vì quét.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tra cứu):** với mỗi `l`, độ dài `r` cần thiết được xác định bởi `suf[r] = x − pre[l]`. Vì các số dương, `suf` tăng nghiêm ngặt ⇒ có **tối đa một** `r` thỏa ⇒ tra bằng `dict` hoặc `bisect`.

**Insight 2 (đảo bài toán):** tìm đoạn giữa **dài nhất** có tổng `target = total − x`. Vì mọi số **dương**, tổng của cửa sổ `[j..i]` **tăng** khi mở rộng phải và **giảm** khi thu hẹp trái ⇒ **cửa sổ trượt**:
- Mở rộng `i` (thêm `nums[i]`).
- **Nếu** tổng `> target` **thì** thu hẹp từ trái cho tới khi `≤ target`.
- **Nếu** tổng `== target` **thì** cập nhật độ dài lớn nhất.

**Trường hợp biên:** `target < 0` (tức `x > total`) ⇒ `−1`; `target == 0` ⇒ đoạn giữa rỗng ⇒ lấy hết `n` phần tử.

**Bất biến cửa sổ:** sau khi xử lý `i`, `j` là chỉ số **nhỏ nhất** sao cho `sum(nums[j..i]) ≤ target`; mọi cửa sổ kết thúc tại `i` có tổng đúng `target` (nếu có) chính là `[j..i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố trái + bảng băm tổng hậu tố

In [8]:
def min_ops_hash_suffix(nums, x):
    n = len(nums)
    suffix_len = {0: 0}
    s = 0
    for r in range(1, n + 1):
        s += nums[n - r]
        suffix_len[s] = r            # số dương => mỗi tổng xuất hiện một lần
    best = float('inf')
    left = 0
    for l in range(n + 1):
        if l > 0:
            left += nums[l - 1]
        if left > x:
            break
        r = suffix_len.get(x - left)
        if r is not None and l + r <= n:
            best = min(best, l + r)
    return best if best != float('inf') else -1

for nums_, x_, exp in examples:
    print(min_ops_hash_suffix(nums_, x_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
5 (kỳ vọng 5 )


Điều kiện `l + r ≤ n` bảo đảm tiền tố và hậu tố **không chồng nhau** (không lấy một phần tử hai lần). **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Tiền tố trái + tìm kiếm nhị phân trên tổng hậu tố

In [9]:
def min_ops_binary(nums, x):
    n = len(nums)
    suf = [0]
    for v in reversed(nums):
        suf.append(suf[-1] + v)        # suf[r] = tổng r phần tử cuối, tăng nghiêm ngặt
    best = float('inf')
    left = 0
    for l in range(n + 1):
        if l > 0:
            left += nums[l - 1]
        need = x - left
        if need < 0:
            break
        r = bisect.bisect_left(suf, need, 0, n - l + 1)
        if r <= n - l and suf[r] == need:
            best = min(best, l + r)
    return best if best != float('inf') else -1

for nums_, x_, exp in examples:
    print(min_ops_binary(nums_, x_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
5 (kỳ vọng 5 )


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Giới hạn `hi = n − l + 1` trong `bisect` để không chồng lên phần tiền tố.

### 7.3 Cách 4 — Đảo bài toán + cửa sổ trượt (lời giải chính)

In [10]:
def min_ops_window(nums, x):
    target = sum(nums) - x
    if target < 0:
        return -1
    best_len = -1
    window = 0
    j = 0
    for i, v in enumerate(nums):
        window += v
        while window > target:          # j <= i được bảo đảm vì target >= 0
            window -= nums[j]
            j += 1
        if window == target:
            best_len = max(best_len, i - j + 1)
    if target == 0:
        best_len = max(best_len, 0)     # đoạn giữa rỗng
    return -1 if best_len == -1 else len(nums) - best_len

for nums_, x_, exp in examples:
    print(min_ops_window(nums_, x_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
5 (kỳ vọng 5 )


Khi `target == 0`, vòng `while` đẩy `window` về `0` với `j = i + 1` ⇒ `i − j + 1 = 0` ⇒ `best_len = 0` cũng được ghi nhận tự nhiên; dòng `max(best_len, 0)` chỉ để dễ đọc. **Độ phức tạp:** mỗi chỉ số vào/ra cửa sổ một lần ⇒ `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.4 Cách 5 — Đảo bài toán + tổng tiền tố + bảng băm

Đoạn `nums[a:b]` có tổng `target` ⇔ `pre[b] − pre[a] = target`. Duyệt `b`, tra `first[pre[b] − target]` = chỉ số `a` **nhỏ nhất** có tiền tố đó. Cách này **không cần** số dương (dùng được cho biến thể có số âm), nhưng tốn `O(n)` bộ nhớ.

In [11]:
def min_ops_prefix_map(nums, x):
    target = sum(nums) - x
    if target < 0:
        return -1
    first = {0: 0}                  # tổng tiền tố -> chỉ số nhỏ nhất (độ dài tiền tố)
    pre = 0
    best_len = -1
    for b, v in enumerate(nums, 1):
        pre += v
        if pre not in first:
            first[pre] = b
        a = first.get(pre - target)
        if a is not None:
            best_len = max(best_len, b - a)
    if target == 0:
        best_len = max(best_len, 0)
    return -1 if best_len == -1 else len(nums) - best_len

for nums_, x_, exp in examples:
    print(min_ops_prefix_map(nums_, x_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
5 (kỳ vọng 5 )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1: `nums = [1,1,4,2,3]`, `total = 11`, `x = 5` ⇒ `target = 6`.

| i | nums[i] | window sau khi thêm | thu hẹp | j | window | == 6? | best_len |
|---|---|---|---|---|---|---|---|
| 0 | 1 | 1 | — | 0 | 1 | | −1 |
| 1 | 1 | 2 | — | 0 | 2 | | −1 |
| 2 | 4 | 6 | — | 0 | 6 | ✔ | 3 |
| 3 | 2 | 8 | bỏ 1, bỏ 1 | 2 | 6 | ✔ | 3 |
| 4 | 3 | 9 | bỏ 4 | 3 | 5 | | 3 |

⇒ đáp án `5 − 3 = 2`.

In [12]:
def trace_window(nums, x):
    target = sum(nums) - x
    print(f"nums={nums}, x={x}, total={sum(nums)}, target={target}")
    if target < 0:
        print("  target < 0 -> -1"); return -1
    best_len, window, j = -1, 0, 0
    for i, v in enumerate(nums):
        window += v
        removed = []
        while window > target:
            removed.append(nums[j]); window -= nums[j]; j += 1
        hit = window == target
        if hit:
            best_len = max(best_len, i - j + 1)
        print(f"  i={i} +{v}: bỏ {removed if removed else '-'}, cửa sổ nums[{j}..{i}]={nums[j:i + 1]}, tổng={window}{'  == target' if hit else ''}, best_len={best_len}")
    ans = -1 if best_len == -1 else len(nums) - best_len
    print("  đáp án:", ans)
    return ans

trace_window([1, 1, 4, 2, 3], 5)

nums=[1, 1, 4, 2, 3], x=5, total=11, target=6
  i=0 +1: bỏ -, cửa sổ nums[0..0]=[1], tổng=1, best_len=-1
  i=1 +1: bỏ -, cửa sổ nums[0..1]=[1, 1], tổng=2, best_len=-1
  i=2 +4: bỏ -, cửa sổ nums[0..2]=[1, 1, 4], tổng=6  == target, best_len=3
  i=3 +2: bỏ [1, 1], cửa sổ nums[2..3]=[4, 2], tổng=6  == target, best_len=3
  i=4 +3: bỏ [4], cửa sổ nums[3..4]=[2, 3], tổng=5, best_len=3
  đáp án: 2


2

In [13]:
trace_window([3, 2, 20, 1, 1, 3], 10)

nums=[3, 2, 20, 1, 1, 3], x=10, total=30, target=20
  i=0 +3: bỏ -, cửa sổ nums[0..0]=[3], tổng=3, best_len=-1
  i=1 +2: bỏ -, cửa sổ nums[0..1]=[3, 2], tổng=5, best_len=-1
  i=2 +20: bỏ [3, 2], cửa sổ nums[2..2]=[20], tổng=20  == target, best_len=1
  i=3 +1: bỏ [20], cửa sổ nums[3..3]=[1], tổng=1, best_len=1
  i=4 +1: bỏ -, cửa sổ nums[3..4]=[1, 1], tổng=2, best_len=1
  i=5 +3: bỏ -, cửa sổ nums[3..5]=[1, 1, 3], tổng=5, best_len=1
  đáp án: 5


5

In [14]:
trace_window([5, 6, 7, 8, 9], 4)
trace_window([2, 3], 5)          # target = 0: lấy hết

nums=[5, 6, 7, 8, 9], x=4, total=35, target=31
  i=0 +5: bỏ -, cửa sổ nums[0..0]=[5], tổng=5, best_len=-1
  i=1 +6: bỏ -, cửa sổ nums[0..1]=[5, 6], tổng=11, best_len=-1
  i=2 +7: bỏ -, cửa sổ nums[0..2]=[5, 6, 7], tổng=18, best_len=-1
  i=3 +8: bỏ -, cửa sổ nums[0..3]=[5, 6, 7, 8], tổng=26, best_len=-1
  i=4 +9: bỏ [5], cửa sổ nums[1..4]=[6, 7, 8, 9], tổng=30, best_len=-1
  đáp án: -1
nums=[2, 3], x=5, total=5, target=0
  i=0 +2: bỏ [2], cửa sổ nums[1..0]=[], tổng=0  == target, best_len=0
  i=1 +3: bỏ [3], cửa sổ nums[2..1]=[], tổng=0  == target, best_len=0
  đáp án: 2


2

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minOperations(self, nums: List[int], x: int) -> int:
        target = sum(nums) - x
        if target < 0:
            return -1
        best_len = -1
        window = j = 0
        for i, v in enumerate(nums):
            window += v
            while window > target:
                window -= nums[j]
                j += 1
            if window == target and i - j + 1 > best_len:
                best_len = i - j + 1
        return -1 if best_len == -1 else len(nums) - best_len

for nums_, x_, exp in examples:
    assert Solution().minOperations(nums_, x_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Quy đổi.** Một dãy thao tác hợp lệ lấy đi đúng một tiền tố `nums[:l]` và một hậu tố `nums[n−r:]` với `l + r ≤ n` (mỗi thao tác bóc một đầu; thứ tự không đổi tập phần tử bị lấy). Tổng lấy đi `= x` ⇔ đoạn còn lại `nums[l:n−r]` có tổng `total − x`. Số thao tác `= n − độ dài đoạn còn lại`. Ngược lại, mọi đoạn liên tiếp có tổng `total − x` đều cho một cách lấy. ⇒ Đáp án `= n − (độ dài lớn nhất của đoạn có tổng target)`, hoặc `−1` nếu không có.

**Cửa sổ trượt đúng vì số dương.** Với `i` cố định, hàm `j ↦ sum(nums[j..i])` **giảm nghiêm ngặt** theo `j`. Nên có tối đa một `j` cho tổng bằng `target`, và đó là `j` nhỏ nhất thỏa `sum ≤ target`. Khi `i` tăng, `j` tối ưu **không giảm** (thêm phần tử dương chỉ làm tổng lớn hơn) ⇒ con trỏ `j` chỉ tiến, không lùi — đúng như vòng `while`. Vì vậy thuật toán xét **mọi** đoạn có thể bằng `target` (mỗi `i` một ứng viên) và lấy dài nhất.

**`target = 0`:** đoạn rỗng có tổng 0; vì mọi `nums[i] ≥ 1`, đoạn khác rỗng không thể có tổng 0 ⇒ `best_len = 0` ⇒ đáp án `n` (lấy hết) — vòng `while` cho `j = i + 1` và `window = 0` nên được ghi nhận.

**`target < 0`:** không đoạn nào có tổng âm ⇒ `−1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi `(l, r)` | `O(n²)` | `O(n)` |
| 2. Băm tổng hậu tố | `O(n)` | `O(n)` |
| 3. Nhị phân trên tổng hậu tố | `O(n log n)` | `O(n)` |
| 4. Cửa sổ trượt | `O(n)` | `O(1)` |
| 5. Tiền tố + băm (đảo bài) | `O(n)` | `O(n)` |

Đo trên `n = 10⁵` (brute chỉ chạy `n = 2000`):

In [16]:
rnd = random.Random(1658)
big = [rnd.randint(1, 10**4) for _ in range(10**5)]
xb = sum(big[:30000]) + sum(big[-20000:])           # chắc chắn có nghiệm
for name, fn in [("hash_suffix", min_ops_hash_suffix), ("binary", min_ops_binary), ("window", min_ops_window), ("prefix_map", min_ops_prefix_map), ("Solution", Solution().minOperations)]:
    t0 = time.perf_counter(); r = fn(big, xb); dt = time.perf_counter() - t0
    print(f"{name:<12}: {r:>6}  {dt * 1000:6.1f} ms")
small = big[:2000]; xs = sum(small[:300]) + sum(small[-500:])
t0 = time.perf_counter(); r = min_ops_brute(small, xs); dt = time.perf_counter() - t0
print(f"brute n=2000: {r} ({dt * 1000:.0f} ms) | window: {min_ops_window(small, xs)}")

hash_suffix :  50000    24.8 ms
binary      :  50000    14.1 ms
window      :  50000     7.1 ms
prefix_map  :  50000    20.8 ms
Solution    :  50000     6.6 ms


brute n=2000: 800 (94 ms) | window: 800

## 12. Bộ test

In [17]:
ALL = {
    "hash_suffix": min_ops_hash_suffix,
    "binary": min_ops_binary,
    "window": min_ops_window,
    "prefix_map": min_ops_prefix_map,
    "Solution": lambda a, b: Solution().minOperations(a, b),
}

def check(nums, x, exp=None, brute=True):
    ref = min_ops_brute(nums, x) if brute else min_ops_window(nums, x)
    for name, fn in ALL.items():
        got = fn(list(nums), x)
        assert got == ref, (name, nums, x, got, ref)
    if exp is not None:
        assert ref == exp, (nums, x, ref, exp)
    return ref

for nums_, x_, exp in examples:
    check(nums_, x_, exp)
print("Ví dụ đề: OK")

check([1], 1, 1)                     # lấy hết (target = 0)
check([1], 2, -1)                    # x > total
check([2, 3], 5, 2)
check([1, 1], 3, -1)
check([5, 1, 1, 5], 10, 2)           # hai đầu
check([1, 1, 1, 1, 1], 3, 3)
check([8828, 9581, 49, 9818, 9974, 9869, 9991, 10000, 10000, 10000, 9999, 9993, 9997, 9997], 128096, 14)   # x == tổng mảng -> lấy hết
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [18]:
rnd = random.Random(2017)
for _ in range(3000):
    n = rnd.randint(1, 12)
    nums_ = [rnd.randint(1, 6) for _ in range(n)]
    if rnd.random() < 0.7:                          # thường chọn x có nghiệm
        l = rnd.randint(0, n); r = rnd.randint(0, n - l)
        x_ = sum(nums_[:l]) + sum(nums_[n - r:])
        x_ = max(1, x_)
    else:
        x_ = rnd.randint(1, 40)
    check(nums_, x_)
print("3000 bộ ngẫu nhiên nhỏ so với brute force: OK")

for _ in range(40):
    n = rnd.randint(1, 3000)
    nums_ = [rnd.randint(1, 10**4) for _ in range(n)]
    x_ = rnd.randint(1, sum(nums_) + 10)
    check(nums_, x_, brute=False)
print("40 bộ cỡ vừa so giữa các cách: OK")

3000 bộ ngẫu nhiên nhỏ so với brute force: OK


40 bộ cỡ vừa so giữa các cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam "lấy đầu lớn hơn"

In [19]:
def min_ops_bug_greedy(nums, x):
    arr = list(nums)
    ops = 0
    while x > 0 and arr:
        if arr[0] >= arr[-1]:
            x -= arr.pop(0)
        else:
            x -= arr.pop()
        ops += 1
    return ops if x == 0 else -1

rnd = random.Random(1)
found = None
while found is None:
    nums_ = [rnd.randint(1, 6) for _ in range(rnd.randint(2, 7))]
    x_ = rnd.randint(1, sum(nums_))
    if min_ops_bug_greedy(nums_, x_) != Solution().minOperations(nums_, x_):
        found = (nums_, x_)
print("phản ví dụ:", found, "| tham lam:", min_ops_bug_greedy(*found), "| đúng:", Solution().minOperations(*found))
print("ví dụ 3:", min_ops_bug_greedy([3, 2, 20, 1, 1, 3], 10), "| đúng:", Solution().minOperations([3, 2, 20, 1, 1, 3], 10))

phản ví dụ: ([6, 2, 4, 6, 1], 17) | tham lam: -1 | đúng: 4
ví dụ 3: -1 | đúng: 5


**Vì sao sai:** lấy đầu lớn hơn có thể làm `x` "nhảy qua" 0 hoặc chọn sai phía; lựa chọn ở mỗi bước ảnh hưởng tới các phần tử sẽ lộ ra sau. **Sửa:** xét mọi cặp `(l, r)` một cách thông minh (cửa sổ trượt / băm).

### Lỗi 2 — Khởi tạo `best_len = 0` thay vì `−1`

In [20]:
def min_ops_bug_init(nums, x):
    target = sum(nums) - x
    if target < 0:
        return -1
    best_len, window, j = 0, 0, 0              # SAI: 0 nghĩa là "đã tìm thấy đoạn rỗng"
    for i, v in enumerate(nums):
        window += v
        while window > target:
            window -= nums[j]; j += 1
        if window == target:
            best_len = max(best_len, i - j + 1)
    return len(nums) - best_len

print("sai :", min_ops_bug_init([5, 6, 7, 8, 9], 4), "| đúng:", Solution().minOperations([5, 6, 7, 8, 9], 4))

sai : 5 | đúng: -1


**Vì sao sai:** `best_len = 0` ứng với "lấy hết mảng", chỉ hợp lệ khi `target == 0`. Khi không đoạn nào đạt `target`, phải trả `−1`. **Sửa:** khởi tạo `−1` (giá trị "chưa tìm thấy").

### Lỗi 3 — Chặn cửa sổ phải khác rỗng (`j ≤ i`) nên bỏ sót `target = 0`

In [21]:
def min_ops_bug_nonempty(nums, x):
    target = sum(nums) - x
    if target < 0:
        return -1
    best_len, window, j = -1, 0, 0
    for i, v in enumerate(nums):
        window += v
        while window > target and j < i:       # SAI: không cho cửa sổ rỗng
            window -= nums[j]; j += 1
        if window == target:
            best_len = max(best_len, i - j + 1)
    return -1 if best_len == -1 else len(nums) - best_len

print("sai :", min_ops_bug_nonempty([1, 1], 2), "| đúng:", Solution().minOperations([1, 1], 2))

sai : -1 | đúng: 2


**Vì sao sai:** khi `x == total` cần lấy **hết** mảng (đoạn giữa rỗng). Chặn `j < i` khiến cửa sổ luôn giữ ít nhất 1 phần tử và tổng không bao giờ về 0. **Sửa:** cho phép `j` vượt `i` (như code nộp), hoặc xử lý riêng `target == 0`.

### Lỗi 4 — Quên điều kiện không chồng nhau `l + r ≤ n` (cách băm hậu tố)

In [22]:
def min_ops_bug_overlap(nums, x):
    n = len(nums)
    suffix_len = {0: 0}; s = 0
    for r in range(1, n + 1):
        s += nums[n - r]; suffix_len[s] = r
    best = float('inf'); left = 0
    for l in range(n + 1):
        if l > 0: left += nums[l - 1]
        r = suffix_len.get(x - left)
        if r is not None:                        # SAI: thiếu l + r <= n
            best = min(best, l + r)
    return best if best != float('inf') else -1

nums_, x_ = [1, 1], 3
print("sai :", min_ops_bug_overlap(nums_, x_), "| đúng:", Solution().minOperations(nums_, x_))

sai : 3 | đúng: -1


**Vì sao sai:** với `[1, 1]`, `x = 3`: `l = 1` (tổng 1) + `r = 2` (tổng 2) = 3 — nhưng phần tử đầu bị tính **hai lần**. **Sửa:** chỉ nhận khi `l + r ≤ n`.

### Lỗi 5 — Trả về độ dài cửa sổ thay vì số thao tác

`return best_len` thay vì `len(nums) − best_len` — đây là lỗi "đảo bài toán xong quên đảo lại". Với ví dụ 1 sẽ trả `3` thay vì `2`.

In [23]:
def min_ops_bug_return(nums, x):
    target = sum(nums) - x
    if target < 0: return -1
    best_len, window, j = -1, 0, 0
    for i, v in enumerate(nums):
        window += v
        while window > target:
            window -= nums[j]; j += 1
        if window == target:
            best_len = max(best_len, i - j + 1)
    return best_len                               # SAI: quên n - best_len

print("sai :", min_ops_bug_return([1, 1, 4, 2, 3], 5), "| đúng:", Solution().minOperations([1, 1, 4, 2, 3], 5))

sai : 3 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lấy từ **hai đầu** mảng" ⇒ phần còn lại là **đoạn giữa liên tiếp** ⇒ đảo sang bài đoạn con.
- "Tối thiểu số phần tử lấy đi" ⇔ "tối đa độ dài đoạn giữ lại".
- Mảng **toàn số dương** + tổng mục tiêu ⇒ **cửa sổ trượt** `O(n)`/`O(1)`.
- Có số âm/0 ⇒ **tổng tiền tố + hash map** (lưu chỉ số **đầu tiên** để có đoạn dài nhất).
- Kiểm tra ngay các biên: `x > total` ⇒ `−1`; `x == total` ⇒ `n`.

## 15. Biến thể của bài

1. **LeetCode 1423 — Maximum Points You Can Obtain from Cards:** lấy đúng `k` lá từ hai đầu, tối đa tổng ⇒ cửa sổ cố định độ dài `n − k` có tổng **nhỏ nhất**.
2. **LeetCode 325 — Maximum Size Subarray Sum Equals k** (có số âm): tiền tố + băm (cách 5).
3. **LeetCode 209 — Minimum Size Subarray Sum:** cửa sổ trượt tìm đoạn **ngắn nhất** có tổng `≥ target`.
4. **Có số âm trong `nums`:** cửa sổ trượt **không còn đúng**; dùng cách 5.
5. **Lấy tối đa `k` lần, tối đa tổng lấy được mà ≤ x:** tiền tố/hậu tố + hai con trỏ.

In [24]:
def max_score_cards(cardPoints, k):
    """LeetCode 1423: tối đa tổng khi lấy đúng k lá từ hai đầu."""
    n = len(cardPoints); w = n - k
    total = sum(cardPoints)
    if w == 0:
        return total
    cur = sum(cardPoints[:w]); best_mid = cur
    for i in range(w, n):
        cur += cardPoints[i] - cardPoints[i - w]
        best_mid = min(best_mid, cur)
    return total - best_mid

def min_ops_with_negatives(nums, x):
    """Biến thể 4: nums có thể âm/0 — tiền tố + băm; không thoát sớm khi target < 0 (đoạn tổng âm vẫn có thể tồn tại)."""
    target = sum(nums) - x
    first = {0: 0}; pre = 0; best_len = -1
    for b, v in enumerate(nums, 1):
        pre += v
        if pre not in first: first[pre] = b
        a = first.get(pre - target)
        if a is not None: best_len = max(best_len, b - a)
    if target == 0: best_len = max(best_len, 0)
    return -1 if best_len == -1 else len(nums) - best_len

def brute_any(nums, x):
    n = len(nums); best = -1
    for l in range(n + 1):
        for r in range(n - l + 1):
            if sum(nums[:l]) + sum(nums[n - r:]) == x and (best == -1 or l + r < best):
                best = l + r
    return best

print("1423 [1,2,3,4,5,6,1], k=3 ->", max_score_cards([1, 2, 3, 4, 5, 6, 1], 3))
rnd = random.Random(9)
for _ in range(1500):
    nums_ = [rnd.randint(-4, 6) for _ in range(rnd.randint(1, 9))]
    x_ = rnd.randint(-10, 15)
    assert min_ops_with_negatives(nums_, x_) == brute_any(nums_, x_)
print("Biến thể có số âm (tiền tố + băm) khớp brute force: OK")

1423 [1,2,3,4,5,6,1], k=3 -> 12
Biến thể có số âm (tiền tố + băm) khớp brute force: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi `(l, r)` | thử trực tiếp | `O(n²)` | `O(n)` | đối chiếu |
| 2. Băm hậu tố | tra `x − pre[l]` | `O(n)` | `O(n)` | tư duy "hai đầu" trực tiếp |
| 3. Nhị phân hậu tố | `suf` tăng dần | `O(n log n)` | `O(n)` | khi quen `bisect` |
| 4. Cửa sổ trượt | đoạn giữa dài nhất, số dương | `O(n)` | `O(1)` | **nộp bài** |
| 5. Tiền tố + băm | đoạn giữa, chỉ số đầu tiên | `O(n)` | `O(n)` | khi có số âm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minOperations([1, 1, 1, 2, 2], 4)` bằng bao nhiêu?

**Bài 2 (tính tay):** Với `nums = [2, 1, 3, 1, 2, 4]`, `x = 6`: tính `target`, liệt kê mọi đoạn có tổng `target`, suy ra đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `best_plan(nums, x)` trả về cặp `(l, r)` (số phần tử lấy bên trái, bên phải) của một phương án tối ưu, hoặc `None`.

**Bài 4 (tự viết lại):** Viết lại cách 2 bằng **hai con trỏ** không dùng `dict`: bắt đầu lấy nhiều phần tử trái nhất có thể (`l` lớn nhất mà `pre[l] ≤ x`), rồi lần lượt giảm `l` và tăng `r`.

**Bài 5 (mini challenge):** Viết `count_ways(nums, x)` đếm **số cặp** `(l, r)` khác nhau (với `l + r ≤ n`) sao cho lấy `l` trái, `r` phải cho tổng đúng `x`, trong `O(n)`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_plan(nums, x):
    # TODO: cửa sổ trượt, lưu (j, i) của đoạn dài nhất; l = j, r = n - 1 - i
    pass

# Bài 4 — TODO
def min_ops_two_pointers(nums, x):
    # TODO: l = số phần tử trái tối đa với tổng <= x; sau đó lặp: thêm phần tử phải, bớt phần tử trái khi tổng > x
    pass

# Bài 5 — TODO
def count_ways(nums, x):
    # TODO: với mỗi đoạn giữa có tổng target (mỗi i tối đa một đoạn), đếm 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `total = 7`, `target = 3`.
- Mức 2: Đoạn tổng 3: `[1,1,1]`, `[1,2]`.
- Mức 3: Dài nhất là 3 ⇒ đáp án `5 − 3`.

**Bài 2**
- Mức 1: `total = 13`, `target = 7`.
- Mức 2: Đoạn tổng 7: `[1,3,1,2]`, `[1,2,4]`, …
- Mức 3: Đoạn dài nhất dài 4 ⇒ đáp án 2.

**Bài 3**
- Mức 1: Đoạn giữa `[j..i]` ⇒ `l = j`, `r = n − 1 − i`.
- Mức 2: Đoạn rỗng (`target = 0`) ⇒ `(n, 0)`.
- Mức 3: Kiểm `sum(nums[:l]) + sum(nums[n−r:]) == x` và `l + r == Solution()`.

**Bài 4**
- Mức 1: Tổng hiện tại `cur = pre[l] + suf[r]`.
- Mức 2: Khi tăng `r` mà `cur > x`, giảm `l` cho tới khi `cur ≤ x` (và `l ≥ 0`).
- Mức 3: Mỗi khi `cur == x` và `l + r ≤ n` thì cập nhật đáp án.

**Bài 5**
- Mức 1: Mỗi cặp `(l, r)` hợp lệ ↔ một đoạn giữa có tổng `target`.
- Mức 2: Số dương ⇒ với mỗi điểm cuối có tối đa một điểm đầu.
- Mức 3: Đừng quên đoạn rỗng: có `n + 1` đoạn rỗng (mỗi vị trí cắt), ứng với `n + 1` cặp `(l, n − l)` khi `target = 0`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
print("Bài 1:", Solution().minOperations([1, 1, 1, 2, 2], 4))
assert Solution().minOperations([1, 1, 1, 2, 2], 4) == 2 == min_ops_brute([1, 1, 1, 2, 2], 4)

# Bài 2
nums2, x2 = [2, 1, 3, 1, 2, 4], 6
t2 = sum(nums2) - x2
segs = all_middle(nums2, t2)
print("Bài 2: target =", t2, "| các đoạn:", [s for _, _, s in segs], "| đáp án:", Solution().minOperations(nums2, x2))
assert Solution().minOperations(nums2, x2) == len(nums2) - max(j - i for i, j, _ in segs)

# Bài 3
def best_plan_sol(nums, x):
    n = len(nums)
    target = sum(nums) - x
    if target < 0:
        return None
    best, window, j = None, 0, 0
    for i, v in enumerate(nums):
        window += v
        while window > target:
            window -= nums[j]; j += 1
        if window == target and (best is None or i - j + 1 > best[1] - best[0]):
            best = (j, i + 1)                    # đoạn giữa nums[j:i+1]
    if best is None:
        return None
    a, b = best
    return a, n - b

rnd = random.Random(3)
for _ in range(2000):
    nums_ = [rnd.randint(1, 5) for _ in range(rnd.randint(1, 10))]
    x_ = rnd.randint(1, 30)
    plan = best_plan_sol(nums_, x_)
    exp = Solution().minOperations(nums_, x_)
    if exp == -1:
        assert plan is None
    else:
        l, r = plan
        assert l + r == exp and sum(nums_[:l]) + sum(nums_[len(nums_) - r:]) == x_
print("Bài 3 OK | ví dụ 3 ->", best_plan_sol([3, 2, 20, 1, 1, 3], 10))

Bài 1: 2
Bài 2: target = 7 | các đoạn: [[2, 1, 3, 1], [1, 3, 1, 2], [1, 2, 4]] | đáp án: 2
Bài 3 OK | ví dụ 3 -> (2, 3)


In [27]:
# Bài 4
def min_ops_two_pointers_sol(nums, x):
    n = len(nums)
    l, cur = 0, 0
    while l < n and cur + nums[l] <= x:
        cur += nums[l]; l += 1
    best = l if cur == x else float('inf')
    for r in range(1, n + 1):
        cur += nums[n - r]
        while cur > x and l > 0:
            l -= 1; cur -= nums[l]
        if cur > x or l + r > n:
            break
        if cur == x:
            best = min(best, l + r)
    return best if best != float('inf') else -1

# Bài 5
def count_ways_sol(nums, x):
    n = len(nums)
    target = sum(nums) - x
    if target < 0:
        return 0
    if target == 0:
        return n + 1
    cnt, window, j = 0, 0, 0
    for i, v in enumerate(nums):
        window += v
        while window > target:
            window -= nums[j]; j += 1
        cnt += window == target
    return cnt

def count_ways_brute(nums, x):
    n = len(nums)
    return sum(sum(nums[:l]) + sum(nums[n - r:]) == x for l in range(n + 1) for r in range(n - l + 1))

rnd = random.Random(4)
for _ in range(3000):
    nums_ = [rnd.randint(1, 5) for _ in range(rnd.randint(1, 10))]
    x_ = rnd.randint(1, 30)
    assert min_ops_two_pointers_sol(nums_, x_) == Solution().minOperations(nums_, x_)
    assert count_ways_sol(nums_, x_) == count_ways_brute(nums_, x_)
print("Bài 4, Bài 5 OK | count_ways([1,1,4,2,3], 5) =", count_ways_sol([1, 1, 4, 2, 3], 5))

Bài 4, Bài 5 OK | count_ways([1,1,4,2,3], 5) = 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lấy từ hai đầu" ⇒ đoạn giữa dài nhất có tổng target (số dương):**

```python
target = sum(nums) - x
if target < 0:
    return -1
best_len, window, j = -1, 0, 0
for i, v in enumerate(nums):
    window += v
    while window > target:
        window -= nums[j]; j += 1
    if window == target:
        best_len = max(best_len, i - j + 1)
return -1 if best_len == -1 else len(nums) - best_len
```

**Template có số âm:** `first = {0: 0}`; duyệt tiền tố, `a = first.get(pre - target)`, chỉ ghi `first[pre]` lần đầu.

**Checklist:**
- [ ] Đổi sang tìm đoạn giữa có tổng `total − x`?
- [ ] `x > total` ⇒ `−1`; `x == total` ⇒ `n` (cửa sổ rỗng được phép)?
- [ ] `best_len` khởi tạo `−1` (chưa tìm thấy)?
- [ ] Trả về `n − best_len`, không phải `best_len`?
- [ ] Nếu dùng tiền tố/hậu tố: không chồng nhau (`l + r ≤ n`)?
- [ ] Đã so với brute force `O(n²)` trên dữ liệu ngẫu nhiên?
- [ ] Chữ ký đúng: `def minOperations(self, nums: List[int], x: int) -> int`?